In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spur_vdd_S5_20261021_170000.csv"; None = newest run of RUN_PREFIXES
RUN_PREFIXES = ("pll_spur_perturb", "pll_spur_vdd")  # spur_perturb, spur_supply_sweep

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for prefix in RUN_PREFIXES for p in PLL_DIR.glob(prefix + "_*.csv")
                         if not p.stem.endswith("_traces")), key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
caps = pd.read_csv(RESULT_CSV).rename(columns={"index": "line"})  # "index" clashes with itertuples
traces = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")).rename(columns={"index": "line"})
FLOOR_BIN, FLOOR_QUANTILE = 50e3, 0.2  # as the script: the floor between the lines
GRID = "#e4e3df"
conditions = caps.drop_duplicates("condition")[["condition", "condition_label", "locked"]].set_index("condition")
n_cond = len(conditions)


def cond_name(n):
    return f"{n}: {conditions.loc[n, 'condition_label']}"


def cond_color(n):
    return sample_colorscale("Turbo", [0.1 + 0.8 * (n - 1) / max(n_cond - 1, 1)])[0]


def save(fig, name):
    fig.update_layout(template="plotly_white", font=dict(size=16))
    fig.write_html(RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}.html"))
    fig.show()


carrier = caps[caps["step"] == "carrier"].set_index("condition")
wide = caps[caps["step"] == "wide"].pivot(index="condition", columns="sideband", values=["hump_offset", "hump_dbc"])
track = caps[caps["step"] == "track"]
ref_offset = track[track["condition"] == track["condition"].min()].set_index("line")["peak_offset"]
summary = pd.DataFrame({
    "label": conditions["condition_label"],
    "locked": conditions["locked"],
    "carrier [dBm]": carrier["carrier_dbm"],
    "hump upper [kHz]": wide[("hump_offset", 1)] / 1e3,
    "hump upper [dBc]": wide[("hump_dbc", 1)],
    "hump lower [kHz]": wide[("hump_offset", -1)] / 1e3,
    "hump lower [dBc]": wide[("hump_dbc", -1)],
})
for line, t in track.groupby("line"):
    t = t.set_index("condition")
    name = f"{ref_offset[line] / 1e3:.0f} kHz"
    summary[f"{name} shift [Hz]"] = t["peak_offset"] - ref_offset[line]
    summary[f"{name} [dBc]"] = t["peak_dbc"]
summary.round(1)

,label,locked,carrier [dBm],hump upper [kHz],hump upper [dBc],hump lower [kHz],hump lower [dBc],556 kHz shift [Hz],556 kHz [dBc],704 kHz shift [Hz],...,1587 kHz shift [Hz],1587 kHz [dBc],1881 kHz shift [Hz],1881 kHz [dBc],1948 kHz shift [Hz],1948 kHz [dBc],2117 kHz shift [Hz],2117 kHz [dBc],2381 kHz shift [Hz],2381 kHz [dBc]
condition,,,,,,,,,,,,,,,,,,,,,
1,reference,1,0.3,2175.0,-54.2,2125.0,-54.5,0.0,-44.3,0.0,...,0.0,-35.7,0.0,-54.2,0.0,-24.4,0.0,-31.2,0.0,-35.4
2,bias to 40uA,1,0.3,2175.0,-54.8,3025.0,-54.0,91.4,-50.6,18.6,...,-34.3,-37.3,-8254.3,-51.3,-52.9,-24.0,-44.3,-30.7,10.0,-33.7
3,ref ampl to 1V,1,0.3,2125.0,-54.5,2175.0,-54.2,71.4,-45.3,22.9,...,-20.0,-34.9,114.3,-52.4,-50.0,-23.8,-30.0,-31.0,-127.1,-36.4
4,ref sine,1,0.3,2175.0,-53.3,2125.0,-52.9,118.6,-46.9,-10.0,...,-82.9,-35.0,-8207.1,-47.1,-58.6,-22.8,74.3,-29.5,-81.4,-34.5
5,cpu supply 0.65,1,0.3,2125.0,-55.4,2125.0,-55.3,147.1,-41.5,1.4,...,-122.9,-35.4,-4.3,-54.0,-94.3,-24.4,-34.3,-31.6,-214.3,-38.9
6,PLL supply 0.65,1,0.5,625.0,-60.0,625.0,-60.2,165.7,-44.3,1.4,...,-204.3,-42.2,21.4,-60.6,-71.4,-32.8,-201.4,-39.8,-188.6,-45.1
7,PLL supply 0.85,1,0.3,2125.0,-55.1,2125.0,-54.5,165.7,-40.8,-14.3,...,-148.6,-34.6,52.9,-53.4,-30.0,-22.9,-125.7,-32.7,-115.7,-38.4
8,IO 1.5,1,-2.2,2125.0,-54.7,2125.0,-54.4,187.1,-44.1,-58.6,...,-62.9,-35.2,80.0,-53.5,-41.4,-24.2,-68.6,-30.9,-85.7,-36.2
9,fpga disconnect,0,0.4,2125.0,-58.5,2125.0,-58.5,178.6,-44.8,-21.4,...,-25.7,-36.9,-57.1,-49.9,-812.9,-33.7,-170.0,-30.6,-158.6,-41.9


In [9]:
# Wide capture of the upper sideband per condition: level relative to the carrier against the offset. One figure per
# condition.
for n in conditions.index:
    c = caps[(caps["condition"] == n) & (caps["step"] == "wide") & (caps["sideband"] == 1)].iloc[0]
    t = traces[(traces["condition"] == n) & (traces["step"] == "wide") & (traces["sideband"] == 1)]
    fig = go.Figure()
    fig.add_scatter(x=(t["freq"] - c["f_carrier"]) / 1e6, y=t["level_dbm"] - c["carrier_dbm"], mode="lines",
                    line=dict(width=1), hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"Upper sideband, condition {cond_name(n)} (RBW {c['rbw']:g} Hz)", height=500)
    fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
    fig.update_yaxes(title=f"level [dBc in {c['rbw']:g} Hz]", gridcolor=GRID)
    save(fig, f"wide_condition{n}")

In [10]:
# The floor between the lines (lower FLOOR_QUANTILE per FLOOR_BIN bin) of the upper sideband, all conditions in one
# plot: does the hump move, grow or shrink?
fig = go.Figure()
for n in conditions.index:
    c = caps[(caps["condition"] == n) & (caps["step"] == "wide") & (caps["sideband"] == 1)].iloc[0]
    t = traces[(traces["condition"] == n) & (traces["step"] == "wide") & (traces["sideband"] == 1)]
    offset = t["freq"].to_numpy() - c["f_carrier"]
    level = t["level_dbm"].to_numpy() - c["carrier_dbm"]
    bins = np.floor(offset / FLOOR_BIN)
    centers = [(b + 0.5) * FLOOR_BIN / 1e6 for b in np.unique(bins)]
    floors = [np.percentile(level[bins == b], 100 * FLOOR_QUANTILE) for b in np.unique(bins)]
    fig.add_scatter(x=centers, y=floors, mode="lines+markers", name=cond_name(n), line=dict(color=cond_color(n)),
                    hovertemplate="%{x:.3f} MHz: %{y:.1f} dBc<extra>" + cond_name(n) + "</extra>")
fig.update_layout(title="Floor between the lines (upper sideband) per condition", height=550,
                  legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
fig.update_yaxes(title=f"floor [dBc in {caps.loc[caps['step'] == 'wide', 'rbw'].iloc[0]:g} Hz]", gridcolor=GRID)
save(fig, "floor_per_condition")

In [11]:
# PLL supply sweep (spur_supply_sweep, labels "PLL VDD x.xxx V"): per tracked line its level, and the hump (floor
# maximum of the upper sideband) against the supply voltage. One plot each.
vdd = conditions["condition_label"].str.extract(r"PLL VDD ([0-9.]+) V", expand=False).astype(float)
if vdd.notna().any():
    fig = go.Figure()
    for line, t in track.groupby("line"):
        t = t.assign(vdd=t["condition"].map(vdd)).sort_values("condition")
        fig.add_scatter(x=t["vdd"], y=t["peak_dbc"], mode="markers+lines", name=f"{ref_offset[line] / 1e3:.0f} kHz",
                        text=t["condition"], hovertemplate="%{x:.3f} V (condition %{text}): %{y:.1f} dBc<extra>"
                                                           + f"{ref_offset[line] / 1e3:.0f} kHz" + "</extra>")
    fig.update_layout(title="Line level vs PLL supply", height=550, legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="PLL supply [V]", gridcolor=GRID)
    fig.update_yaxes(title="level [dBc in 100 Hz]", gridcolor=GRID)
    save(fig, "line_level_vs_vdd")

    hump = caps[(caps["step"] == "wide") & (caps["sideband"] == 1)].assign(vdd=lambda d: d["condition"].map(vdd))
    fig = go.Figure()
    fig.add_scatter(x=hump["vdd"], y=hump["hump_dbc"], mode="markers", marker=dict(size=10), text=hump["condition"],
                    hovertemplate="%{x:.3f} V (condition %{text}): %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title="Hump (floor maximum, upper sideband) vs PLL supply", height=500)
    fig.update_xaxes(title="PLL supply [V]", gridcolor=GRID)
    fig.update_yaxes(title=f"floor maximum [dBc in {hump['rbw'].iloc[0]:g} Hz]", gridcolor=GRID)
    save(fig, "hump_vs_vdd")

In [ ]:
# Tracked lines per condition: the shift of each line's offset from its offset in the first condition (does a line
# move?), and its level. One plot each, one trace per line.
for col, title, ytitle, name in (("shift", "Line shift from the first condition", "shift [Hz]", "line_shift"),
                                 ("peak_dbc", "Line level", "level [dBc in 100 Hz]", "line_level")):
    fig = go.Figure()
    for line, t in track.groupby("line"):
        t = t.sort_values("condition")
        y = t["peak_offset"] - ref_offset[line] if col == "shift" else t[col]
        fig.add_scatter(x=[cond_name(n) for n in t["condition"]], y=y, mode="lines+markers",
                        name=f"{ref_offset[line] / 1e3:.0f} kHz",
                        hovertemplate="%{x}: %{y:.1f}<extra>" + f"{ref_offset[line] / 1e3:.0f} kHz" + "</extra>")
    fig.update_layout(title=title, height=550, legend=dict(orientation="h", y=-0.3))
    fig.update_xaxes(title="condition", gridcolor=GRID)
    fig.update_yaxes(title=ytitle, gridcolor=GRID)
    save(fig, name)

: 